# 10.02 端云双模式对话应用

## 本节目标

- 在开发板加载 DeepSeek-R1-Distill-Qwen-1.5B-FP16，并完成端侧流式生成。
- 通过 SSH 本地转发调用 10.01 中启动的云端 MindIE 服务。
- 在一个 Gradio 页面中切换端侧与云端后端，记录两条推理路径的对话体验。

## 实验原理

### 端云推理路径

本册运行在开发板上。端侧模式在开发板内存中调用 MindSpore 模型；云端模式将请求送到开发板本机端口，再经 SSH 隧道到达云端 MindIE。

| 模式 | 模型位置 | 前端调用方式 |
| --- | --- | --- |
| 端侧 | 开发板内存 | Gradio 直接调用 MindSpore 模型 |
| 云端 | 云端 MindIE 服务 | Gradio → 开发板 <code>127.0.0.1:1025</code> → SSH 隧道 → 云端 MindIE |

页面维护一份聊天界面状态。切换按钮清空聊天记录，新一轮消息由所选模型接收。运行本册前，完成 10.01 的云端服务准备，并通过 <code>/v1/models</code> 接口检查。

## 实验环境

### 运行目录和 Kernel

端侧运行目录由第一个代码单元中的 <code>LAB_DIR</code> 定义。本 Notebook、模型缓存、日志和 PEM 文件位于该目录。首次打开时，选择 <code>/usr/local/miniconda3</code> 的 Base Conda Kernel 并运行下一节；它会注册 <code>Lab 10 Edge (base Conda)</code>，Kernel 名称为 <code>lab10-edge</code>。随后在 Jupyter 的 Kernel 菜单中切换到 <code>Lab 10 Edge (base Conda)</code>，重启 Kernel，再从第一节运行。

Jupyter Kernel 是 Notebook 代码实际运行的 Python 进程。这里复用开发板镜像提供的 Base Conda，使 MindSpore、MindNLP、Gradio 与 Ascend 运行时处于同一环境。

### 云端访问密钥

PEM 文件与本 Notebook 并列。本文用 <code>KeyPair-*.pem</code> 表示课程发放的密钥文件名，<code>*</code> 位置填写实际后缀，例如 <code>KeyPair-abc123.pem</code>。端侧启动 JupyterLab 前，通过 <code>LAB10_CLOUD_KEY_PATH</code> 指定 PEM 的绝对路径；Notebook 使用该路径建立 SSH 隧道。

~~~bash
export LAB10_CLOUD_KEY_PATH="<LAB_DIR>/KeyPair-<实际后缀>.pem"
jupyter lab
~~~

PEM 文件权限为 <code>600</code>。模型、日志、PID 和主机指纹写入 Lab 10 目录的子目录。

## 实验流程

依次准备端侧环境、下载模型、建立 SSH 隧道并启动 Gradio 页面。先完成一条端侧短问答，再切换到云端；记录首轮中的模型加载、JIT 编译、隧道建立和网络请求时间。

### 1. 准备端侧环境

开发板复用已有的 Base Conda。MindSpore、MindNLP、Gradio 和 ipykernel 由基础镜像提供；该单元在 Base Conda 中确认 <code>requests</code> 可用。端侧模型由 MindNLP 访问 Modelers 镜像。

CANN 将 MindSpore 与 Ascend 运行时、设备库连接起来。单元加载 CANN 的 <code>set_env.sh</code>，并确认 MindSpore 的设备目标为 Ascend。Python、MindSpore、MindNLP 与 Gradio 的实际版本写入 <code>state/edge_environment.json</code>，作为本次实验的环境记录。

注册 <code>lab10-edge</code> 后，Jupyter 的 Kernel 菜单会显示 Base Conda 解释器。

In [ ]:
from __future__ import annotations

import importlib.metadata
import importlib.util
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

BASE_CONDA = Path('/usr/local/miniconda3')
LAB_DIR = Path('/home/HwHiAiUser/workspace/10_end-cloud_dual-mode_dialogue_system')
MODEL_CACHE = LAB_DIR / 'models'
MODEL_DIR = MODEL_CACHE / 'model' / 'MindSpore-Lab' / 'DeepSeek-R1-Distill-Qwen-1.5B-FP16'
LOG_DIR = LAB_DIR / 'logs'
STATE_DIR = LAB_DIR / 'state'

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

def uses_base_conda() -> bool:
    executable = Path(sys.executable).resolve()
    return str(executable).startswith(str(BASE_CONDA) + os.sep)

require(BASE_CONDA.is_dir(), f'找不到 Base Conda：{BASE_CONDA}')
require(uses_base_conda(), '当前 Kernel 不属于 /usr/local/miniconda3。请先选择 Base Conda Kernel，运行本单元格后再切换到 lab10-edge。')

LAB_DIR.mkdir(parents=True, exist_ok=True)
for directory in (MODEL_CACHE, LOG_DIR, STATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def source_ascend_environment() -> Path | None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    seen: set[Path] = set()
    for script in candidates:
        if not script.is_file() or script in seen:
            continue
        seen.add(script)
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        return script
    return None

ascend_script = source_ascend_environment()
if ascend_script:
    print('已加载 CANN 环境：', ascend_script)
else:
    print('未找到 set_env.sh，继续检查当前 Kernel 是否已经带有 Ascend 运行时。')

required_runtime_modules = ['mindspore', 'mindnlp', 'gradio', 'ipykernel']
missing_runtime = [name for name in required_runtime_modules if importlib.util.find_spec(name) is None]
require(not missing_runtime, 'Base Conda 缺少运行时包：' + '、'.join(missing_runtime) + '。请使用课程提供的开发板镜像。')

pip_packages = {'requests': 'requests'}
missing_pip = [distribution for module, distribution in pip_packages.items() if importlib.util.find_spec(module) is None]
if missing_pip:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_pip])
    print('已安装：', '、'.join(missing_pip))
else:
    print('Python 依赖已齐全。')

subprocess.check_call([
    sys.executable, '-m', 'ipykernel', 'install', '--user',
    '--name', 'lab10-edge', '--display-name', 'Lab 10 Edge (base Conda)',
])

import mindspore

mindspore.set_context(
    enable_graph_kernel=True,
    mode=mindspore.GRAPH_MODE,
    jit_config={'jit_level': 'O2'},
)
require(mindspore.get_context('device_target') == 'Ascend', '当前 MindSpore 设备不是 Ascend。请检查开发板 NPU 与 CANN 环境。')

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'unknown'

environment = {
    'python': sys.version.split()[0],
    'python_executable': sys.executable,
    'conda_prefix': sys.prefix,
    'mindspore': getattr(mindspore, '__version__', 'unknown'),
    'mindnlp': package_version('mindnlp'),
    'gradio': package_version('gradio'),
    'model_cache': str(MODEL_CACHE),
    'device_target': mindspore.get_context('device_target'),
    'lab_dir': str(LAB_DIR),
}
(STATE_DIR / 'edge_environment.json').write_text(json.dumps(environment, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(environment, ensure_ascii=False, indent=2))
print('Kernel 已注册。请切换到 lab10-edge 后，从本单元格重新运行。')


### 2. 下载端侧模型

模型来自 Modelers 的 <code>MindSpore-Lab/DeepSeek-R1-Distill-Qwen-1.5B-FP16</code> 仓库。对话时，分词器将文本编码为 token；模型读取 FP16 权重生成 token，再由分词器解码为文字。

MindNLP 将下载内容缓存到 <code>models/model/MindSpore-Lab/DeepSeek-R1-Distill-Qwen-1.5B-FP16</code>。模型缓存、日志和实验记录位于同一 Lab 10 目录。<code>EDGE_MODEL_REVISION</code> 使用 <code>None</code>，单元在状态文件中记录下载的文件名和大小。

In [ ]:
EDGE_MODEL_ID = 'MindSpore-Lab/DeepSeek-R1-Distill-Qwen-1.5B-FP16'
EDGE_MODEL_REVISION = None
os.environ['MINDNLP_CACHE'] = str(MODEL_CACHE)

from mindnlp.transformers import AutoModelForCausalLM, AutoTokenizer

def model_files(path: Path) -> list[Path]:
    return sorted(file for file in path.rglob('*') if file.is_file())

def has_model_weights(path: Path) -> bool:
    return any(file.suffix in {'.safetensors', '.ckpt', '.bin'} for file in model_files(path))

load_options = {'cache_dir': str(MODEL_CACHE), 'mirror': 'modelers'}
if EDGE_MODEL_REVISION is not None:
    load_options['revision'] = EDGE_MODEL_REVISION

tokenizer = AutoTokenizer.from_pretrained(EDGE_MODEL_ID, **load_options)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = 'left'
model = AutoModelForCausalLM.from_pretrained(
    EDGE_MODEL_ID,
    ms_dtype=mindspore.float16,
    low_cpu_mem_usage=True,
    **load_options,
)

require((MODEL_DIR / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_DIR}')
require(has_model_weights(MODEL_DIR), f'模型目录没有找到权重文件：{MODEL_DIR}')
manifest = [
    {'path': str(file.relative_to(MODEL_DIR)), 'bytes': file.stat().st_size}
    for file in model_files(MODEL_DIR)
]
model_record = {
    'model_id': EDGE_MODEL_ID,
    'revision': EDGE_MODEL_REVISION,
    'model_dir': str(MODEL_DIR),
    'files': manifest,
}
(STATE_DIR / 'edge_model_files.json').write_text(json.dumps(model_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'模型文件：{len(manifest)} 个，记录已写入 {STATE_DIR / "edge_model_files.json"}')


### 3. 加载模型并完成 JIT 编译

上一节加载 FP16 权重后，这一节将模型切换到推理状态并进行 JIT 编译。图模式先将计算过程编译为可在 Ascend 上执行的图，首轮提问包含编译等待；后续形状相近的请求主要复用已编译的图。

模型状态写入 <code>state/edge_model_state.json</code>。首轮和后续请求分开记录：首轮包含 JIT 编译时间，后续请求主要反映持续生成的推理耗时。

In [ ]:
import threading

import numpy as np
from mindnlp.configs import set_pyboost
from mindnlp.core import ops
from mindnlp.transformers import StaticCache

model.set_train(False)
model.jit()
set_pyboost(False)
edge_generation_lock = threading.Lock()

model_state = {
    'model_id': EDGE_MODEL_ID,
    'model_dir': str(MODEL_DIR),
    'dtype': str(model.dtype),
    'device_target': mindspore.get_context('device_target'),
    'jit_level': 'O2',
}
(STATE_DIR / 'edge_model_state.json').write_text(json.dumps(model_state, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(model_state, ensure_ascii=False, indent=2))


### 4. 实现端侧流式生成

自回归生成每次产生一个 token。模型在生成时读取前文的注意力键和值；<code>StaticCache</code> 将这部分结果保存在内存中，下一步计算沿用已有缓存。

~~~text
提示词预填充 → 保存 K/V → 生成一个 token → 复用缓存 → 生成下一个 token
~~~

本实验的缓存上限为 768 个 token，输入最多使用 511 个 token，生成最多使用 256 个 token，满足 <code>511 + 256 + 1 ≤ 768</code>。聊天记录从左侧截断，较新的对话会保留。token 是分词器处理后的单位，中文字符数与 token 数通常不同。每轮对话创建独立缓存。

In [ ]:
SYSTEM_PROMPT = '你是课程实验中的对话助手。回答直接、简短。'
MAX_CACHE_TOKENS = 768
MAX_NEW_TOKENS = 256
MAX_INPUT_TOKENS = MAX_CACHE_TOKENS - MAX_NEW_TOKENS - 1
TEMPERATURE = 0.7
TOP_P = 0.9

def top_p_sample(probabilities, top_p: float = TOP_P):
    probabilities_np = probabilities.asnumpy()
    sorted_indices = np.argsort(-probabilities_np, axis=-1)
    sorted_probabilities = np.take_along_axis(probabilities_np, sorted_indices, axis=-1)
    cumulative_probabilities = np.cumsum(sorted_probabilities, axis=-1)
    sorted_probabilities[cumulative_probabilities - sorted_probabilities > top_p] = 0.0
    denominators = np.sum(sorted_probabilities, axis=-1, keepdims=True)
    sorted_probabilities = sorted_probabilities / np.maximum(denominators, 1e-12)
    sorted_probabilities_tensor = mindspore.Tensor(sorted_probabilities, dtype=mindspore.float32)
    sorted_indices_tensor = mindspore.Tensor(sorted_indices, dtype=mindspore.int32)
    sampled_index = ops.multinomial(sorted_probabilities_tensor, 1)
    return mindspore.ops.gather(sorted_indices_tensor, sampled_index, axis=1, batch_dims=1)

@mindspore.jit(jit_config=mindspore.JitConfig(jit_syntax_level='STRICT'))
def decode_one_token_logits(model, current_token, cache_position, past_key_values):
    return model(
        current_token,
        position_ids=None,
        cache_position=cache_position,
        past_key_values=past_key_values,
        return_dict=False,
        use_cache=True,
    )[0]

def next_token_from_logits(logits):
    if TEMPERATURE > 0:
        probabilities = mindspore.mint.softmax(logits[:, -1] / TEMPERATURE, dim=-1)
        return top_p_sample(probabilities)
    return mindspore.mint.argmax(logits[:, -1], dim=-1)[:, None]

def edge_messages(chat_history: list[tuple[str, str]]) -> list[dict[str, str]]:
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for user_text, assistant_text in chat_history:
        messages.append({'role': 'user', 'content': user_text})
        if assistant_text:
            messages.append({'role': 'assistant', 'content': assistant_text})
    return messages

def edge_answer_stream(chat_history: list[tuple[str, str]]):
    messages = edge_messages(chat_history)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    encoded = tokenizer(
        [prompt],
        return_tensors='ms',
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    model_inputs = {key: value for key, value in encoded.items() if key in {'input_ids', 'attention_mask'}}
    model_inputs['input_ids'] = model_inputs['input_ids'].to(mindspore.int32)
    batch_size, sequence_length = model_inputs['input_ids'].shape
    require(batch_size == 1, '端侧页面一次只处理一条消息。')

    past_key_values = StaticCache(
        config=model.config,
        max_batch_size=batch_size,
        max_cache_len=MAX_CACHE_TOKENS,
        dtype=model.dtype,
    )
    cache_position = ops.arange(sequence_length)
    with edge_generation_lock:
        logits = model(
            **model_inputs,
            cache_position=cache_position,
            past_key_values=past_key_values,
            return_dict=False,
            use_cache=True,
        )[0]
        next_token = next_token_from_logits(logits)
        stop_token_ids = set(getattr(tokenizer, 'all_special_ids', []))
        if tokenizer.eos_token_id is not None:
            stop_token_ids.add(tokenizer.eos_token_id)
        generated_ids: list[int] = []
        decode_position = mindspore.tensor([sequence_length + 1], dtype=mindspore.int32)
        for token_index in range(MAX_NEW_TOKENS):
            token_id = int(next_token.asnumpy().reshape(-1)[0])
            if token_id in stop_token_ids:
                break
            generated_ids.append(token_id)
            yield tokenizer.decode(generated_ids, skip_special_tokens=True)
            if token_index + 1 == MAX_NEW_TOKENS:
                break
            next_logits = decode_one_token_logits(model, next_token, decode_position, past_key_values)
            next_token = next_token_from_logits(next_logits)
            decode_position += 1


### 5. 建立云端 SSH 隧道

隧道将开发板的 <code>127.0.0.1:1025</code> 转发到云端 MindIE 的 <code>127.0.0.1:1025</code>。云端模式下，Gradio 请求开发板本机端口。

~~~text
开发板上的请求
  http://127.0.0.1:1025/v1/chat/completions
             │
             └── SSH -L ──► 云端 127.0.0.1:1025/v1/chat/completions
~~~

首次连接时，<code>state/cloud_known_hosts</code> 保存云端主机指纹；PEM 文件保存客户端私钥。隧道日志记录密钥路径、PID 和连接输出。隧道建立后，Notebook 请求 <code>/v1/models</code>，并检查返回列表中的 <code>qwen2-7b-instruct</code>。

In [ ]:
import atexit
import signal
import socket
import stat
import time

import requests

CLOUD_SSH_HOST = 'dev-modelarts.cn-southwest-2.huaweicloud.com'
CLOUD_SSH_PORT = 31133
CLOUD_SSH_USER = 'ma-user'
CLOUD_MODEL_NAME = 'qwen2-7b-instruct'
CLOUD_KEY_PATH = Path(
    os.environ.get('LAB10_CLOUD_KEY_PATH', str(LAB_DIR / 'KeyPair-d7ad.pem'))
).expanduser()
CLOUD_LOCAL_HOST = '127.0.0.1'
CLOUD_LOCAL_PORT = 1025
CLOUD_API_BASE = f'http://{CLOUD_LOCAL_HOST}:{CLOUD_LOCAL_PORT}'
CLOUD_READINESS_PATH = '/v1/models'
TUNNEL_PID_FILE = STATE_DIR / 'cloud_tunnel.pid'
TUNNEL_LOG_FILE = LOG_DIR / 'cloud_tunnel.log'
KNOWN_HOSTS_FILE = STATE_DIR / 'cloud_known_hosts'

def port_is_open(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def read_pid(path: Path) -> int | None:
    try:
        return int(path.read_text(encoding='utf-8').strip())
    except (OSError, ValueError):
        return None

def process_is_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def process_command(pid: int) -> str:
    cmdline = Path(f'/proc/{pid}/cmdline')
    try:
        return cmdline.read_bytes().replace(b'\0', b' ').decode(errors='replace')
    except OSError:
        return ''

def is_lab_tunnel(pid: int) -> bool:
    command = process_command(pid)
    forward = f'{CLOUD_LOCAL_HOST}:{CLOUD_LOCAL_PORT}:127.0.0.1:1025'
    return 'ssh' in command and CLOUD_SSH_HOST in command and forward in command

def tunnel_is_healthy() -> bool:
    try:
        response = requests.get(f'{CLOUD_API_BASE}{CLOUD_READINESS_PATH}', timeout=3)
        if response.status_code != 200:
            return False
        return isinstance(response.json().get('data'), list)
    except (requests.RequestException, ValueError):
        return False

def tail_log(path: Path, limit: int = 2000) -> str:
    if not path.is_file():
        return ''
    return path.read_text(encoding='utf-8', errors='replace')[-limit:]

def stop_lab_tunnel(quiet: bool = False) -> None:
    pid = read_pid(TUNNEL_PID_FILE)
    if pid is None:
        return
    if not process_is_alive(pid):
        TUNNEL_PID_FILE.unlink(missing_ok=True)
        return
    if not is_lab_tunnel(pid):
        raise RuntimeError(f'PID 文件中的进程不是本实验创建的 SSH 隧道：{pid}')
    os.kill(pid, signal.SIGTERM)
    deadline = time.monotonic() + 5
    while process_is_alive(pid) and time.monotonic() < deadline:
        time.sleep(0.1)
    if process_is_alive(pid):
        os.kill(pid, signal.SIGKILL)
    TUNNEL_PID_FILE.unlink(missing_ok=True)
    if not quiet:
        print('已关闭 SSH 隧道。')

def list_cloud_models() -> list[str]:
    response = requests.get(f'{CLOUD_API_BASE}/v1/models', timeout=10)
    response.raise_for_status()
    payload = response.json()
    return [item['id'] for item in payload.get('data', []) if isinstance(item, dict) and item.get('id')]

def start_or_reuse_tunnel() -> None:
    require(CLOUD_KEY_PATH.is_file(), f'找不到 PEM 文件：{CLOUD_KEY_PATH}')
    key_mode = stat.S_IMODE(CLOUD_KEY_PATH.stat().st_mode)
    require(key_mode & 0o077 == 0, f'PEM 权限应为 600，当前为 {oct(key_mode)}。请执行 chmod 600 {CLOUD_KEY_PATH}')
    require(shutil.which('ssh') is not None, '找不到 ssh 命令。')

    old_pid = read_pid(TUNNEL_PID_FILE)
    if old_pid is not None and process_is_alive(old_pid):
        require(is_lab_tunnel(old_pid), f'PID 文件中的进程不是本实验创建的 SSH 隧道：{old_pid}')
        if tunnel_is_healthy():
            print(f'复用 SSH 隧道，PID={old_pid}')
            return
        stop_lab_tunnel(quiet=True)
    elif old_pid is not None:
        TUNNEL_PID_FILE.unlink(missing_ok=True)

    require(not port_is_open(CLOUD_LOCAL_HOST, CLOUD_LOCAL_PORT), f'{CLOUD_LOCAL_HOST}:{CLOUD_LOCAL_PORT} 已被占用，无法创建本实验的 SSH 隧道。')
    command = [
        'ssh', '-N', '-i', str(CLOUD_KEY_PATH), '-p', str(CLOUD_SSH_PORT),
        '-o', 'BatchMode=yes',
        '-o', 'IdentitiesOnly=yes',
        '-o', 'ExitOnForwardFailure=yes',
        '-o', 'StrictHostKeyChecking=accept-new',
        '-o', f'UserKnownHostsFile={KNOWN_HOSTS_FILE}',
        '-o', 'ServerAliveInterval=30',
        '-o', 'ServerAliveCountMax=3',
        '-L', f'{CLOUD_LOCAL_HOST}:{CLOUD_LOCAL_PORT}:127.0.0.1:1025',
        f'{CLOUD_SSH_USER}@{CLOUD_SSH_HOST}',
    ]
    with TUNNEL_LOG_FILE.open('a', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    TUNNEL_PID_FILE.write_text(f'{process.pid}\n', encoding='utf-8')
    deadline = time.monotonic() + 30
    while time.monotonic() < deadline:
        if process.poll() is not None:
            TUNNEL_PID_FILE.unlink(missing_ok=True)
            raise RuntimeError(f'SSH 隧道启动失败。日志：{tail_log(TUNNEL_LOG_FILE)}')
        if tunnel_is_healthy():
            print(f'SSH 隧道已就绪，PID={process.pid}，本地地址为 {CLOUD_API_BASE}')
            return
        time.sleep(0.5)
    stop_lab_tunnel(quiet=True)
    raise RuntimeError('SSH 已连接，但云端 /v1/models 在 30 秒内没有返回有效结果。请检查云端 MindIE 服务。')

start_or_reuse_tunnel()
cloud_models = list_cloud_models()
require(CLOUD_MODEL_NAME in cloud_models, f'云端没有模型 {CLOUD_MODEL_NAME}；接口返回：{cloud_models}')
print('云端模型：', cloud_models)

if not globals().get('_lab10_tunnel_atexit_registered', False):
    atexit.register(lambda: stop_lab_tunnel(quiet=True))
    _lab10_tunnel_atexit_registered = True


### 6. 启动端云双模式对话页面

端侧模式直接调用本 Notebook 中常驻的模型。云端模式经由本地 SSH 隧道调用 MindIE 的 OpenAI 兼容接口。两种模式都将回答逐段追加到同一个聊天窗口；页面绑定开发板的 <code>0.0.0.0:7860</code>，局域网浏览器通过开发板 IP 和端口 7860 访问页面。

切换按钮改变后端函数并创建新的聊天记录。连续追问使用当前模型的上下文。页面初始使用端侧模式；用同一问题切换后再次提问，记录两次响应。

云端回答使用 <code>stream=true</code> 调用 <code>/v1/chat/completions</code>。服务逐段返回内容，Gradio 在收到新内容时更新聊天记录。

In [ ]:
import gradio as gr

APP_HOST = '0.0.0.0'
APP_PORT = 7860

def cloud_answer_stream(chat_history: list[tuple[str, str]]):
    payload = {
        'model': CLOUD_MODEL_NAME,
        'messages': edge_messages(chat_history),
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_tokens': MAX_NEW_TOKENS,
        'stream': True,
    }
    response = requests.post(
        f'{CLOUD_API_BASE}/v1/chat/completions',
        json=payload,
        stream=True,
        timeout=(10, 300),
    )
    try:
        response.raise_for_status()
        answer = ''
        for line in response.iter_lines(decode_unicode=False):
            if not line or not line.startswith(b'data:'):
                continue
            data = line[5:].strip()
            if data == b'[DONE]':
                break
            event = json.loads(data.decode('utf-8'))
            choices = event.get('choices', [])
            if not choices:
                continue
            delta = choices[0].get('delta') or {}
            content = delta.get('content') or ''
            if content:
                answer += content
                yield answer
    finally:
        response.close()

def backend_name(backend: str) -> str:
    return '云端' if backend == 'cloud' else '端侧'

def submit_message(message: str, history: list[tuple[str, str]], backend: str):
    text = (message or '').strip()
    if not text:
        yield '', history or []
        return
    updated_history = list(history or [])
    updated_history.append((text, ''))
    yield '', updated_history
    try:
        stream = cloud_answer_stream(updated_history) if backend == 'cloud' else edge_answer_stream(updated_history)
        received = False
        for partial_answer in stream:
            received = True
            updated_history[-1] = (text, partial_answer)
            yield '', updated_history
        if not received:
            updated_history[-1] = (text, '模型没有返回可显示的内容。')
            yield '', updated_history
    except Exception as exc:
        updated_history[-1] = (text, f'请求失败：{type(exc).__name__}: {exc}')
        yield '', updated_history

def switch_backend(backend: str):
    target = 'cloud' if backend == 'edge' else 'edge'
    return (
        target,
        [],
        f'当前：{backend_name(target)}',
        gr.update(value=f'切换到{backend_name(backend)}'),
    )

def clear_history():
    return []

old_demo = globals().get('demo')
if old_demo is not None:
    old_demo.close()
require(not port_is_open('127.0.0.1', APP_PORT), f'端口 {APP_PORT} 已被占用。请关闭旧的 Gradio 页面后重试。')

with gr.Blocks(title='Lab 10 端云双模对话') as demo:
    backend_state = gr.State('edge')
    status = gr.Markdown('当前：端侧')
    chat = gr.Chatbot(label='对话', height=500, type='tuples')
    message = gr.Textbox(label='输入消息', placeholder='输入内容后按 Enter 或点击发送')
    with gr.Row():
        send = gr.Button('发送', variant='primary')
        clear = gr.Button('清空')
        switch = gr.Button('切换到云端')

    send.click(submit_message, inputs=[message, chat, backend_state], outputs=[message, chat], api_name=False)
    message.submit(submit_message, inputs=[message, chat, backend_state], outputs=[message, chat], api_name=False)
    clear.click(clear_history, outputs=chat, queue=False, api_name=False)
    switch.click(
        switch_backend,
        inputs=backend_state,
        outputs=[backend_state, chat, status, switch],
        queue=False,
        api_name=False,
    )

demo.queue(default_concurrency_limit=1)
launch_result = demo.launch(
    server_name=APP_HOST,
    server_port=APP_PORT,
    share=False,
    show_error=True,
    prevent_thread_lock=True,
)
print(f'Gradio 已启动：开发板局域网地址的 {APP_PORT} 端口。')


### 7. 关闭页面与 SSH 隧道

运行下一节会关闭 Gradio 页面和本 Notebook 记录的 SSH 隧道。模型在 Kernel 重启后释放；下载的模型文件和实验记录保留在 Lab 10 目录。

In [ ]:
if globals().get('demo') is not None:
    demo.close()
    print('Gradio 页面已关闭。')
stop_lab_tunnel()


## 本节总结

端侧模型在开发板的 Gradio 进程中推理，云端模型通过 SSH 隧道调用。切换后会产生新的聊天记录，可分别记录两个后端的回答和响应时间。

使用同一问题完成首轮问答和一次追问，记录模型、后端、首轮耗时、后续耗时与上下文衔接情况。